1.	To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the text corpus and find most similar words of “King”.

(Two separate codes 1. Without using any pretrained model 2. Using Pretrained model (Gensim) )


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
import os

# Path to the user's corpus
file_path = "/content/drive/MyDrive/CNLP-Lab/Experiment-6/glove_corpus.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        corpus = [line.strip() for line in f if line.strip()]
    print(f"Successfully loaded {len(corpus)} lines from {file_path}.")
else:
    print(f"File not found at {file_path}. Falling back to sample corpus.")
    corpus = [
        "the king rules the kingdom",
        "the queen rules the kingdom",
        "the king is a brave man",
        "the queen is a beautiful woman",
        "the prince is the son of the king and queen",
        "royal family consists of king queen prince and princess"
    ]

# Tokenization and vocabulary building
tokens = [word.lower() for sentence in corpus for word in sentence.split()]
vocab = list(set(tokens))
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

# Build co-occurrence matrix
window_size = 2
co_occurrence = defaultdict(lambda: defaultdict(float))

for sentence in corpus:
    words = sentence.lower().split()
    for i, word in enumerate(words):
        word_id = word_to_id[word]
        start = max(0, i - window_size)
        end = min(len(words), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                context_id = word_to_id[words[j]]
                distance = abs(i - j)
                co_occurrence[word_id][context_id] += 1.0 / distance

# Prepare training data
x_i = []
x_j = []
x_ij = []
for i in range(vocab_size):
    for j in range(vocab_size):
        if co_occurrence[i][j] > 0:
            x_i.append(i)
            x_j.append(j)
            x_ij.append(co_occurrence[i][j])

x_i = torch.LongTensor(x_i)
x_j = torch.LongTensor(x_j)
x_ij = torch.FloatTensor(x_ij)

# Define GloVe Model in PyTorch
class GloVe(nn.Module):
    def __init__(self, vocab_size, embed_size=50, x_max=100, alpha=0.75):
        super(GloVe, self).__init__()
        self.w = nn.Embedding(vocab_size, embed_size)
        self.w_tilde = nn.Embedding(vocab_size, embed_size)
        self.b = nn.Embedding(vocab_size, 1)
        self.b_tilde = nn.Embedding(vocab_size, 1)
        self.x_max = x_max
        self.alpha = alpha

        # Initialize weights
        init_range = 0.5 / embed_size
        self.w.weight.data.uniform_(-init_range, init_range)
        self.w_tilde.weight.data.uniform_(-init_range, init_range)
        self.b.weight.data.zero_()
        self.b_tilde.weight.data.zero_()

    def forward(self, i, j, x_ij):
        w_i = self.w(i)
        w_j_tilde = self.w_tilde(j)
        b_i = self.b(i).squeeze()
        b_j_tilde = self.b_tilde(j).squeeze()

        # Weighting function
        f_x = torch.where(x_ij < self.x_max, (x_ij / self.x_max) ** self.alpha, torch.ones_like(x_ij))

        # Loss calculation
        inner_product = torch.sum(w_i * w_j_tilde, dim=1)
        loss = f_x * (inner_product + b_i + b_j_tilde - torch.log(x_ij)) ** 2
        return torch.mean(loss)

# Train the model
model = GloVe(vocab_size, embed_size=10)
optimizer = optim.Adagrad(model.parameters(), lr=0.05)

for epoch in range(200):
    optimizer.zero_grad()
    loss = model(x_i, x_j, x_ij)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/200, Loss: {loss.item():.4f}")

# Extract final embeddings (W + W_tilde as suggested in GloVe paper)
embeddings = model.w.weight.data.numpy() + model.w_tilde.weight.data.numpy()

def get_cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def find_most_similar_scratch(word, top_n=3):
    word = word.lower()
    if word not in word_to_id:
        return f"Word '{word}' not in vocabulary."
    word_vec = embeddings[word_to_id[word]]
    similarities = []
    for other_word, idx in word_to_id.items():
        if other_word != word:
            sim = get_cosine_similarity(word_vec, embeddings[idx])
            similarities.append((other_word, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

# Find similar words to 'king'
print("\nMost similar to 'king' (Trained from Scratch):")
print(find_most_similar_scratch('king'))

Successfully loaded 24 lines from /content/drive/MyDrive/CNLP-Lab/Experiment-6/glove_corpus.txt.
Vocabulary size: 73
Epoch 50/200, Loss: 0.0000
Epoch 100/200, Loss: 0.0000
Epoch 150/200, Loss: 0.0000
Epoch 200/200, Loss: 0.0000

Most similar to 'king' (Trained from Scratch):
[('queen', np.float32(0.97357583)), ('the', np.float32(0.9478392)), ('rules', np.float32(0.9259702))]


### Part 2: Using Pretrained GloVe Model with Gensim

In [3]:
!pip install gensim

import gensim.downloader as api

# Load pretrained GloVe embeddings (Wikipedia 2014 + Gigaword 5, 50 dimensions)
print("Loading pretrained GloVe model...")
glove_model = api.load("glove-wiki-gigaword-50")

# Find the most similar words to 'king'
similar_words = glove_model.most_similar('king', topn=5)

print("\nMost similar to 'king' (Pretrained GloVe):")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 72.2 MB/s  0:00:00
Loading pretrained GloVe model...
[==================================================] 100.0% 66.0/66.0MB downloaded

Most similar to 'king' (Pretrained GloVe):
prince: 0.8236
queen: 0.7839
ii: 0.7746
emperor: 0.7736
son: 0.7667


2.	To implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus (shown below).

In [4]:
import re
from collections import defaultdict, Counter

# Provided text corpus
corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

def preprocess_and_tokenize(text):
    # Lowercase, clean punctuation, and tokenize
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()
    # Add start and end tags for sentence boundaries
    return ['<s>'] + tokens + ['</s>']

# Tokenize entire corpus
tokenized_corpus = [preprocess_and_tokenize(sentence) for sentence in corpus]
all_tokens = [token for sentence in tokenized_corpus for token in sentence]
vocab = set(all_tokens)
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print("Sample tokenized sentence:", tokenized_corpus[0])

Vocabulary size: 36
Sample tokenized sentence: ['<s>', 'natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', '</s>']


In [5]:
class NGramLM:
    def __init__(self, tokenized_corpus, n=2):
        self.n = n
        self.vocab = set(token for sentence in tokenized_corpus for token in sentence)
        self.vocab_size = len(self.vocab)
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = defaultdict(int)
        self.build_model(tokenized_corpus)

    def build_model(self, tokenized_corpus):
        for sentence in tokenized_corpus:
            for i in range(len(sentence) - self.n + 1):
                ngram = tuple(sentence[i:i + self.n])
                context = ngram[:-1]
                target = ngram[-1]

                self.ngram_counts[context][target] += 1
                self.context_counts[context] += 1

    def get_next_word_probability(self, context, target, alpha=1.0):
        context = tuple(context[-(self.n-1):])
        context_count = self.context_counts.get(context, 0)
        target_count = self.ngram_counts[context].get(target, 0)

        # Laplace smoothing formula
        probability = (target_count + alpha) / (context_count + alpha * self.vocab_size)
        return probability

    def predict_next_word(self, input_sequence):
        # Preprocess input
        tokens = input_sequence.lower().strip().split()
        context = tuple(tokens[-(self.n-1):])

        # Calculate probability for all words in the vocabulary
        predictions = {}
        for word in self.vocab:
            if word not in ['<s>']:
                predictions[word] = self.get_next_word_probability(context, word)

        # Sort predictions by probability
        sorted_predictions = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        return sorted_predictions

# Initialize Bigram (n=2) and Trigram (n=3) models
bigram_model = NGramLM(tokenized_corpus, n=2)
trigram_model = NGramLM(tokenized_corpus, n=3)

# Testing prediction
test_phrase = "natural language"
predictions = bigram_model.predict_next_word(test_phrase)

print(f"Top 5 predictions for next word after '{test_phrase}':")
for word, prob in predictions[:5]:
    print(f"{word}: {prob:.4f}")

Top 5 predictions for next word after 'natural language':
processing: 0.0930
models: 0.0698
model: 0.0465
</s>: 0.0465
artificial: 0.0233
